# 🤖 AI Stock Predictor - Google Colab

Complete setup and usage guide for the AI Stock Predictor system in Google Colab.

## Features:
- 📊 Data Collection (Price & News)
- 🔑 API Key Setup (.env file)
- 📈 Data Visualization
- 📰 News Analysis
- 🤖 Model Training
- 🔮 Price Predictions

---

## 🚀 Quick Start

1. **Enable GPU**: Runtime → Change runtime type → GPU (T4 or better)
2. **Run all cells** in order
3. **Upload files** when prompted:
   - `ai_stock_predictor.py` (required)
   - `.env` file (required - for API keys)
4. **Configure** your symbol and settings
5. **Collect data** and train models

---

## 1. Install Required Packages

In [ ]:
# Install all required packages
%pip install -q torch pandas numpy yfinance requests plotly python-dotenv transformers accelerate

print("✅ All packages installed!")

## 2. Upload Files

**Upload these 2 files (both required):**
1. `ai_stock_predictor.py`
2. `.env` (required - contains your API keys). Run the cell below until both files are present.

In [ ]:
# Upload files to Colab - BOTH files are required
from google.colab import files
import os

# Accept .env or filenames that end with .env (some systems strip the leading dot)
def has_env(u):
    if '.env' in u:
        return True
    for k in u:
        if k.strip().lower().endswith('.env') or k.strip() == 'env':
            return True
    return False

uploaded = {}
while True:
    print("📤 Upload these 2 files (both required):")
    print("   1. ai_stock_predictor.py")
    print("   2. .env")
    print("\nClick 'Choose Files', select both, then 'Upload'.\n")
    new_uploads = files.upload()
    uploaded.update(new_uploads)

    has_py = 'ai_stock_predictor.py' in uploaded
    has_env_file = has_env(uploaded)

    if has_py:
        print("✅ ai_stock_predictor.py")
    else:
        print("❌ Missing: ai_stock_predictor.py")
    if has_env_file:
        print("✅ .env")
    else:
        print("❌ Missing: .env")

    if has_py and has_env_file:
        print("\n✅ Both required files are present. Continue to the next cell.")
        break
    print("\n⚠️ Upload the missing file(s) and run this cell again, or choose files again now.")

## 3. Import Libraries and Load API Keys

In [ ]:
# Import libraries
import os
import sys
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Load .env file if uploaded
try:
    from dotenv import load_dotenv
    load_dotenv()  # This loads .env from current directory
    print("✅ .env file loaded (if it exists)")
except ImportError:
    print("⚠️ python-dotenv not installed")
except Exception as e:
    print(f"💡 .env file not found or error: {e}")

# Load API keys from Colab Secrets (alternative to .env)
try:
    from google.colab import userdata
    print("\n🔑 Loading API keys from Colab Secrets...")
    keys_loaded = 0
    api_keys = [
        'NEWSAPI_KEY', 'ALPHAVANTAGE_API_KEY', 'FINNHUB_API_KEY',
        'POLYGON_API_KEY', 'TWITTER_BEARER_TOKEN', 'REDDIT_CLIENT_ID',
        'NEWSCATCHER_API_KEY', 'BING_SEARCH_API_KEY'
    ]
    for key_name in api_keys:
        try:
            key_value = userdata.get(key_name)
            if key_value:
                os.environ[key_name] = str(key_value).strip()
                keys_loaded += 1
                print(f"  ✅ {key_name} loaded from Secrets")
        except:
            pass
    
    if keys_loaded > 0:
        print(f"  ✅ Loaded {keys_loaded} API key(s) from Colab Secrets")
    else:
        print("  💡 No keys found in Secrets - using .env file or free sources only")
    print("  💡 To add keys: Click 🔑 icon (Secrets) in Colab sidebar")
except ImportError:
    print("💡 Not in Colab - using .env file or environment variables")

# Import the AI Stock Predictor
try:
    from ai_stock_predictor import (
        AIStockPredictor, 
        TimeFrame, 
        MarketRegime,
        Prediction
    )
    print("\n✅ Libraries imported successfully!")
except ImportError as e:
    print(f"\n❌ Import error: {e}")
    print("💡 Make sure ai_stock_predictor.py is uploaded in the cell above")

# Check GPU
import torch
if torch.cuda.is_available():
    print(f"\n✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("\n⚠️ No GPU detected - training will be slow")
    print("   Enable GPU: Runtime → Change runtime type → GPU")

In [ ]:
# Check which API keys are loaded
api_keys_status = {
    'NewsAPI': ('NEWSAPI_KEY', 'https://newsapi.org/register', '100/day free'),
    'Alpha Vantage': ('ALPHAVANTAGE_API_KEY', 'https://www.alphavantage.co/support/#api-key', '500/day free'),
    'Finnhub': ('FINNHUB_API_KEY', 'https://finnhub.io/register', '60/min free'),
    'Polygon.io': ('POLYGON_API_KEY', 'https://polygon.io/', 'Paid'),
    'Twitter/X': ('TWITTER_BEARER_TOKEN', 'https://developer.twitter.com/', 'Limited free'),
    'Reddit': ('REDDIT_CLIENT_ID', 'https://www.reddit.com/prefs/apps', 'Free'),
    'NewsCatcher': ('NEWSCATCHER_API_KEY', 'https://newscatcher.ai/', '100/month free'),
    'Bing News': ('BING_SEARCH_API_KEY', 'https://www.microsoft.com/en-us/bing/apis/bing-news-search-api', 'Azure account'),
}

keys_found = []
keys_missing = []

for service_name, (key_name, url, tier) in api_keys_status.items():
    key_value = os.getenv(key_name, '')
    if key_value:
        masked_key = key_value[:4] + "..." + key_value[-4:] if len(key_value) > 8 else "***"
        keys_found.append((service_name, masked_key, tier))
    else:
        keys_missing.append((service_name, url, tier))

print("🔑 API Key Status:")
print("=" * 60)

if keys_found:
    print(f"\n✅ {len(keys_found)} API key(s) loaded:")
    for service_name, masked_key, tier in keys_found:
        print(f"  • {service_name} ({tier}) - Key: {masked_key}")

if keys_missing:
    print(f"\n⚠️ {len(keys_missing)} API key(s) missing:")
    for service_name, url, tier in keys_missing[:5]:
        print(f"  • {service_name} - Get key: {url} ({tier})")
    if len(keys_missing) > 5:
        print(f"  ... and {len(keys_missing) - 5} more")

print("\n💡 Free sources (no API key needed):")
print("  • yfinance (Yahoo Finance)")
print("  • CryptoCompare (for crypto)")

if keys_found:
    print("\n🎉 You have API keys! You can collect 200-4000+ articles.")
else:
    print("\n⚠️ No API keys found. You'll only get ~50 articles from free sources.")
    print("   Options:")
    print("   1. Upload .env file in the upload cell above")
    print("   2. Use Colab Secrets (🔑 icon in sidebar)")

## 5. Configuration

**Change these values to customize your analysis:**

In [ ]:
# Configuration - Change these values as needed
SYMBOL = "BTC-USD"  # Change this: "ETH-USD", "AAPL", "TSLA", etc.
PERIOD = "6mo"  # Options: "1mo", "3mo", "6mo", "1y", "2y"
INTERVAL = "1d"  # Options: "1h", "1d", "1wk"
NEWS_DAYS = 30  # Number of days to look back for news
MAX_ARTICLES_PER_SOURCE = 1000  # Maximum articles per source

# Training settings
EPOCHS = 30  # Number of training epochs (reduce for faster testing)
BATCH_SIZE = 64  # Batch size (increase if GPU memory allows)
LEARNING_RATE = 0.001  # Learning rate

print(f"📊 Configuration:")
print(f"  Symbol: {SYMBOL}")
print(f"  Period: {PERIOD}, Interval: {INTERVAL}")
print(f"  News Lookback: {NEWS_DAYS} days")
print(f"  Max Articles per Source: {MAX_ARTICLES_PER_SOURCE}")
print(f"  Training: {EPOCHS} epochs, batch_size={BATCH_SIZE}, lr={LEARNING_RATE}")

## 6. Initialize Predictor

In [ ]:
# Initialize the AI Stock Predictor
predictor = AIStockPredictor(SYMBOL)
print(f"✅ Predictor initialized for {SYMBOL}")
print(f"   Device: {predictor.device}")

## 7. Collect Data

**This will collect price data, news articles, and calculate features.**

In [ ]:
# Collect all data (price, news, technical indicators, etc.)
print("📊 Collecting data...")
print("This may take a few minutes depending on your internet connection and API limits.\n")

features = predictor.collect_data(
    period=PERIOD,
    interval=INTERVAL,
    news_days=NEWS_DAYS,
    max_news_per_source=MAX_ARTICLES_PER_SOURCE
)

print(f"\n✅ Data collection complete!")
print(f"  Price data points: {len(predictor.price_data)}")
print(f"  News articles: {len(predictor.news_agent.news_cache)}")
print(f"  Features: {len(features.columns)}")

# Show collection statistics
stats = predictor.news_agent.collection_stats
print(f"\n📰 News Sources: {stats.get('sources_used', [])}")
print(f"  Total articles: {len(predictor.news_agent.news_cache)}")
print(f"  Duplicates removed: {stats.get('duplicates_removed', 0)}")

## 8. Visualize Price Data

In [ ]:
# Display price data summary
price_data = predictor.price_data
print(f"Price Data Shape: {price_data.shape}")
print(f"Date Range: {price_data['datetime'].iloc[0]} to {price_data['datetime'].iloc[-1]}")
print(f"\nFirst few rows:")
display(price_data.head())

In [ ]:
# Plot price chart
fig = go.Figure()

# Candlestick chart
fig.add_trace(go.Candlestick(
    x=price_data['datetime'],
    open=price_data['open'],
    high=price_data['high'],
    low=price_data['low'],
    close=price_data['close'],
    name='Price'
))

fig.update_layout(
    title=f'{SYMBOL} Price Chart',
    xaxis_title='Date',
    yaxis_title='Price',
    height=500,
    template='plotly_white'
)

fig.show()

## 9. Analyze News Data

In [ ]:
# Convert news data to DataFrame
news_data = predictor.news_agent.news_cache
if news_data:
    news_df = pd.DataFrame(news_data)
    print(f"News Data Shape: {news_df.shape}")
    print(f"\nColumns: {news_df.columns.tolist()}")
    print(f"\nFirst few articles:")
    display(news_df[['timestamp', 'title', 'source', 'category', 'sentiment']].head(10))
else:
    print("No news data available")

In [ ]:
# News statistics by source
if news_data:
    from collections import Counter
    sources = [item.get('source', 'Unknown') for item in news_data]
    source_counts = Counter(sources)
    
    # Bar chart of articles by source
    fig = go.Figure(data=[
        go.Bar(
            x=list(source_counts.keys())[:15],
            y=list(source_counts.values())[:15],
            marker_color='lightblue'
        )
    ])
    
    fig.update_layout(
        title='News Articles by Source',
        xaxis_title='Source',
        yaxis_title='Number of Articles',
        height=400,
        template='plotly_white',
        xaxis_tickangle=-45
    )
    
    fig.show()
    
    print("\n📰 Articles by source:")
    for source, count in source_counts.most_common(10):
        print(f"  • {source}: {count} articles")

In [ ]:
# Sentiment analysis
if news_data:
    news_df = pd.DataFrame(news_data)
    if 'sentiment' in news_df.columns:
        # Sentiment distribution
        fig = go.Figure(data=[
            go.Histogram(
                x=news_df['sentiment'],
                nbinsx=20,
                marker_color='steelblue'
            )
        ])
        
        fig.update_layout(
            title='News Sentiment Distribution',
            xaxis_title='Sentiment Score',
            yaxis_title='Number of Articles',
            height=400,
            template='plotly_white'
        )
        
        fig.show()
        
        # Sentiment summary
        positive = len(news_df[news_df['sentiment'] > 0.1])
        negative = len(news_df[news_df['sentiment'] < -0.1])
        neutral = len(news_df[(news_df['sentiment'] >= -0.1) & (news_df['sentiment'] <= 0.1)])
        
        print(f"\n📊 Sentiment Summary:")
        print(f"  Positive: {positive} ({positive/len(news_df)*100:.1f}%)")
        print(f"  Neutral: {neutral} ({neutral/len(news_df)*100:.1f}%)")
        print(f"  Negative: {negative} ({negative/len(news_df)*100:.1f}%)")

## 10. Feature Analysis

In [ ]:
# Display features
features = predictor.features
if features is not None:
    print(f"Features Shape: {features.shape}")
    print(f"\nFeature Categories:")
    
    # Categorize features
    technical_features = [col for col in features.columns if any(x in col.lower() for x in ['rsi', 'macd', 'bollinger', 'ema', 'sma', 'atr', 'stoch'])]
    time_features = [col for col in features.columns if any(x in col.lower() for x in ['hour', 'day', 'month', 'week', 'lag'])]
    news_features = [col for col in features.columns if any(x in col.lower() for x in ['news', 'sentiment'])]
    macro_features = [col for col in features.columns if any(x in col.lower() for x in ['fear', 'greed', 'dxy', 'macro'])]
    other_features = [col for col in features.columns if col not in technical_features + time_features + news_features + macro_features and col not in ['datetime', 'open', 'high', 'low', 'close', 'volume']]
    
    print(f"  Technical Indicators: {len(technical_features)}")
    print(f"  Time & Lag Features: {len(time_features)}")
    print(f"  News & Sentiment: {len(news_features)}")
    print(f"  Macro Features: {len(macro_features)}")
    print(f"  Other Features: {len(other_features)}")
    
    print("\nSample features:")
    display(features.head())
else:
    print("No features available")

## 11. Train Models (Optional)

**Uncomment the cell below to train models. This may take 10-30 minutes on GPU.**

In [ ]:
# Train models (uncomment to train)
# print("🤖 Training models (this may take 10-30 minutes on GPU)...")
# predictor.train_models(
#     epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
#     lr=LEARNING_RATE
# )
# print("✅ Training complete!")

## 12. Make Predictions (Optional)

**Requires trained models. Uncomment to make predictions.**

In [ ]:
# Make predictions (uncomment if models are trained)
# if predictor.models_trained:
#     print("🔮 Making predictions...")
#     prediction = predictor.predict(timeframe=TimeFrame.MEDIUM_TERM)
#     
#     print(f"\n📊 Prediction Results:")
#     print(f"  Predicted Price: ${prediction.price:.2f}")
#     print(f"  Confidence: {prediction.confidence:.2%}")
#     print(f"  Price Range: ${prediction.lower_bound:.2f} - ${prediction.upper_bound:.2f}")
#     print(f"  Timeframe: {prediction.timeframe.value}")
#     print(f"  Market Regime: {prediction.regime.value}")
#     print(f"\n  Model Contributions:")
#     for model, contrib in prediction.model_contributions.items():
#         print(f"    {model}: {contrib:.2%}")
# else:
#     print("⚠️ Models not trained yet. Train models first.")

## 13. Export Data

**Export collected data to CSV files.**

In [ ]:
# Export data to CSV
export_dir = "exports"
os.makedirs(export_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Export price data
if predictor.price_data is not None:
    price_file = f"{export_dir}/price_data_{SYMBOL.replace('-', '_')}_{timestamp}.csv"
    predictor.price_data.to_csv(price_file, index=False)
    print(f"✅ Price data exported to: {price_file}")

# Export news data
if predictor.news_agent.news_cache:
    news_df = pd.DataFrame(predictor.news_agent.news_cache)
    news_file = f"{export_dir}/news_data_{SYMBOL.replace('-', '_')}_{timestamp}.csv"
    news_df.to_csv(news_file, index=False)
    print(f"✅ News data exported to: {news_file}")

# Export features
if predictor.features is not None:
    features_file = f"{export_dir}/features_{SYMBOL.replace('-', '_')}_{timestamp}.csv"
    predictor.features.to_csv(features_file, index=False)
    print(f"✅ Features exported to: {features_file}")

# Download files in Colab
try:
    from google.colab import files
    print("\n💡 To download files, run:")
    print(f"   files.download('{price_file}')")
    print(f"   files.download('{news_file}')")
    print(f"   files.download('{features_file}')")
except ImportError:
    print("\n💡 Files saved locally. In Colab, use files.download() to download.")

---

## 📝 Notes

- **API Keys**: 
  - Upload `.env` file in Section 2 (easiest)
  - Or use Colab Secrets (🔑 icon) - Recommended
- **Free Sources**: yfinance and CryptoCompare work without API keys
- **Training**: Model training is optional and can take a long time
- **Predictions**: Requires trained models
- **GPU**: Enable GPU for faster training (Runtime → Change runtime type → GPU)

---

## 🔄 To Use with Different Symbol

Just change `SYMBOL` in Section 5 and re-run the cells from Section 6 onwards!

---